In [43]:
###  import  and  load  the  data sets    

In [44]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/crop_yield.csv")
print("Shape:", df.shape)
df.head()    

Shape: (1000000, 10)


,Region,Soil_Type,Crop,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Weather_Condition,Days_to_Harvest,Yield_tons_per_hectare
0,West,Sandy,Cotton,897.077239,27.676966,False,True,Cloudy,122,6.555816
1,South,Clay,Rice,992.673282,18.026142,True,True,Rainy,140,8.527341
2,North,Loam,Barley,147.998025,29.794042,False,False,Sunny,106,1.127443
3,North,Sandy,Soybean,986.866331,16.644190,False,True,Rainy,146,6.517573
4,South,Silt,Wheat,730.379174,31.620687,True,True,Cloudy,110,7.248251


In [45]:
###  check   the  missing  values    
print("Total missing values:", df.isnull().sum().sum())    

Total missing values: 0


In [46]:
### chek the  dupliactions    
print("Duplicate rows:", df.duplicated().sum())   

Duplicate rows: 0


In [47]:
###  chek   the data  types   
df.info()   

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 10 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   Region                  1000000 non-null  str    
 1   Soil_Type               1000000 non-null  str    
 2   Crop                    1000000 non-null  str    
 3   Rainfall_mm             1000000 non-null  float64
 4   Temperature_Celsius     1000000 non-null  float64
 5   Fertilizer_Used         1000000 non-null  bool   
 6   Irrigation_Used         1000000 non-null  bool   
 7   Weather_Condition       1000000 non-null  str    
 8   Days_to_Harvest         1000000 non-null  int64  
 9   Yield_tons_per_hectare  1000000 non-null  float64
dtypes: bool(2), float64(3), int64(1), str(4)
memory usage: 62.9 MB


In [48]:
###  convert  the  categorical  dat   into  proper  categorical   data    
categorical_cols = ["Region", "Soil_Type", "Crop", "Weather_Condition"]

for col in categorical_cols:
    df[col] = df[col].astype("category")

df[categorical_cols].dtypes    

Region               category
Soil_Type            category
Crop                 category
Weather_Condition    category
dtype: object

In [49]:
print(df["Fertilizer_Used"].dtype)
print(df["Irrigation_Used"].dtype)
df["Fertilizer_Used"] = df["Fertilizer_Used"].astype(bool)
df["Irrigation_Used"] = df["Irrigation_Used"].astype(bool)    

bool
bool


In [50]:
###  get   the  value  counts for  the  categorical  data    
for col in categorical_cols:
    print(f"\n{col}:")
    print(df[col].value_counts())   


Region:
Region
North    250173
West     250074
South    250054
East     249699
Name: count, dtype: int64

Soil_Type:
Soil_Type
Sandy     167119
Loam      166795
Chalky    166779
Silt      166672
Clay      166352
Peaty     166283
Name: count, dtype: int64

Crop:
Crop
Maize      166824
Rice       166792
Barley     166777
Wheat      166673
Cotton     166585
Soybean    166349
Name: count, dtype: int64

Weather_Condition:
Weather_Condition
Sunny     333790
Rainy     333561
Cloudy    332649
Name: count, dtype: int64


In [51]:
print("Yield_tons_per_hectare summary:")
print(df["Yield_tons_per_hectare"].describe())

negative_yield_count = (df["Yield_tons_per_hectare"] < 0).sum()
print(f"\nRows with negative yield (physically impossible): {negative_yield_count}")

df[df["Yield_tons_per_hectare"] < 0].head(10)

Yield_tons_per_hectare summary:
count    1000000.000000
mean           4.649472
std            1.696572
min           -1.147613
25%            3.417637
50%            4.651808
75%            5.879200
max            9.963372
Name: Yield_tons_per_hectare, dtype: float64

Rows with negative yield (physically impossible): 231


,Region,Soil_Type,Crop,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Weather_Condition,Days_to_Harvest,Yield_tons_per_hectare
756,East,Peaty,Cotton,101.019421,33.804131,False,False,Rainy,117,-0.007103
7799,South,Chalky,Rice,108.804894,18.004082,False,False,Sunny,86,-0.061283
8421,East,Chalky,Soybean,168.120735,38.473430,False,False,Cloudy,111,-0.119911
9553,North,Silt,Wheat,156.607973,16.610257,False,False,Sunny,129,-0.193093
15435,South,Chalky,Maize,177.481344,27.719742,False,False,Sunny,94,-0.009811
19517,North,Clay,Maize,107.845638,21.855075,False,False,Rainy,118,-0.265207
22552,East,Sandy,Barley,136.608277,16.723985,False,False,Sunny,101,-0.235296
25967,West,Sandy,Wheat,143.407729,16.252040,False,False,Rainy,77,-0.023612
33285,West,Sandy,Rice,103.921766,16.314536,False,False,Cloudy,80,-0.060267
34902,East,Chalky,Soybean,144.431171,37.817570,False,False,Sunny,112,-0.028898


### Outlier detection (IQR method) on numeric columns   

In [52]:
numerical_cols = ["Rainfall_mm", "Temperature_Celsius", "Days_to_Harvest", "Yield_tons_per_hectare"]

outlier_summary = {}

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outlier_summary[col] = {
        "count": len(outliers),
        "lower_bound": lower_bound,
        "upper_bound": upper_bound
    }

outlier_df = pd.DataFrame(outlier_summary).T
outlier_df

,count,lower_bound,upper_bound
Rainfall_mm,0.0,-349.880055,1449.509665
Temperature_Celsius,0.0,2.506355,52.501415
Days_to_Harvest,0.0,14.500000,194.500000
Yield_tons_per_hectare,84.0,-0.274707,9.571544


###  handle  the  negitive  values    


In [53]:
# Crop yield cannot physically be negative — these are treated as data entry/measurement errors.
# Given only ~84 rows out of 1,000,000 (0.0084%), the safest approach is to remove them
# rather than impute, since imputation could introduce bias into a target variable.

rows_before = df.shape[0]
df = df[df["Yield_tons_per_hectare"] >= 0].copy()
rows_after = df.shape[0]

print(f"Rows removed (negative yield): {rows_before - rows_after}")
print(f"Remaining rows: {rows_after}")

Rows removed (negative yield): 231
Remaining rows: 999769


In [54]:
###  check  the  outlier   again     


In [55]:
Q1 = df["Yield_tons_per_hectare"].quantile(0.25)
Q3 = df["Yield_tons_per_hectare"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

remaining_outliers = df[(df["Yield_tons_per_hectare"] < lower_bound) | (df["Yield_tons_per_hectare"] > upper_bound)]
print(f"Remaining outliers in Yield_tons_per_hectare: {len(remaining_outliers)}")    

Remaining outliers in Yield_tons_per_hectare: 28


In [56]:
print("Remaining outlier rows (by IQR):")
print(remaining_outliers[["Rainfall_mm", "Temperature_Celsius", "Fertilizer_Used",
                           "Irrigation_Used", "Days_to_Harvest", "Yield_tons_per_hectare"]]
      .sort_values("Yield_tons_per_hectare", ascending=False))

print("\nUpper bound was:", upper_bound)
print("Max yield in full cleaned dataset:", df['Yield_tons_per_hectare'].max())

Remaining outlier rows (by IQR):
        Rainfall_mm  Temperature_Celsius  Fertilizer_Used  Irrigation_Used  \
868145   980.537954            37.263468             True             True   
713988   977.575295            32.934272             True             True   
905016   874.775592            27.909656             True             True   
942403   987.281001            23.821032             True             True   
465022   978.875813            35.301038             True             True   
469963   967.070702            15.765251             True             True   
866796   980.380130            26.385247             True             True   
572954   933.096993            39.445050             True             True   
11596    957.380991            36.324216             True             True   
385993   977.530313            30.647480             True             True   
401635   937.648660            39.507361             True             True   
668399   999.780316            

###  sanity   check  the  numarical  data   

In [57]:
print("Rainfall_mm range:", df["Rainfall_mm"].min(), "-", df["Rainfall_mm"].max())
print("Temperature_Celsius range:", df["Temperature_Celsius"].min(), "-", df["Temperature_Celsius"].max())
print("Days_to_Harvest range:", df["Days_to_Harvest"].min(), "-", df["Days_to_Harvest"].max())
print("Yield_tons_per_hectare range:", df["Yield_tons_per_hectare"].min(), "-", df["Yield_tons_per_hectare"].max())    

Rainfall_mm range: 100.00089622522204 - 999.998098221668
Temperature_Celsius range: 15.000034141430271 - 39.99999662316004
Days_to_Harvest range: 60 - 149
Yield_tons_per_hectare range: 0.0004108724039286 - 9.963372228814649


###  save  the  cleaned  data sets    

In [58]:
df.to_csv("../data/cleaned_data.csv", index=False)
print("Cleaned data saved to ../data/cleaned_data.csv")
print("Final shape:", df.shape)  

Cleaned data saved to ../data/cleaned_data.csv
Final shape: (999769, 10)


###  save   the  summary  of  the  data cleaning    

In [59]:
import json

cleaning_summary = {
    "original_shape": [1000000, 10],
    "final_shape": list(df.shape),
    "missing_values_found": 0,
    "duplicates_found": 0,
    "negative_yield_rows_removed": int(rows_before - rows_after),
    "outliers_before_cleaning": {col: int(v["count"]) for col, v in outlier_summary.items()},
    "outliers_after_negative_removal": 28,
    "outliers_retained": 28,
    "categorical_columns_typed": categorical_cols,
    "boolean_columns_typed": ["Fertilizer_Used", "Irrigation_Used"],
    "notes": (
        "Dataset had no missing values or duplicates. Found and removed 84 rows with "
        "physically impossible negative yield values (data entry/measurement errors). "
        "After removal, 28 statistical outliers (IQR method) remained in the target "
        "variable, all representing legitimately high yields under favorable conditions "
        "(high rainfall, fertilizer used, irrigation used) rather than data errors. "
        "These were retained since they reflect real, explainable high-performing farm "
        "scenarios valuable for decision-support recommendations."
    )
}

with open("../reports/data_cleaning_summary.json", "w") as f:
    json.dump(cleaning_summary, f, indent=4)

cleaning_summary

{'original_shape': [1000000, 10],
 'final_shape': [999769, 10],
 'missing_values_found': 0,
 'duplicates_found': 0,
 'negative_yield_rows_removed': 231,
 'outliers_before_cleaning': {'Rainfall_mm': 0,
  'Temperature_Celsius': 0,
  'Days_to_Harvest': 0,
  'Yield_tons_per_hectare': 84},
 'outliers_after_negative_removal': 28,
 'outliers_retained': 28,
 'categorical_columns_typed': ['Region',
  'Soil_Type',
  'Crop',
  'Weather_Condition'],
 'boolean_columns_typed': ['Fertilizer_Used', 'Irrigation_Used'],
 'notes': 'Dataset had no missing values or duplicates. Found and removed 84 rows with physically impossible negative yield values (data entry/measurement errors). After removal, 28 statistical outliers (IQR method) remained in the target variable, all representing legitimately high yields under favorable conditions (high rainfall, fertilizer used, irrigation used) rather than data errors. These were retained since they reflect real, explainable high-performing farm scenarios valuable fo